In [3]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import time
import csv
from pathlib import Path
import google.generativeai as genai
from dotenv import load_dotenv


def setup_environment():
    """Carga las variables de entorno y configura la API de Gemini."""
    load_dotenv(override=True)
    gemini_api_key = os.environ.get("GEMINI_API_KEY")
    print(gemini_api_key[:1] + "..." + gemini_api_key[-3:])
    if not gemini_api_key:
        raise EnvironmentError("GEMINI_API_KEY no está definido en las variables de entorno.")
    genai.configure(api_key=gemini_api_key)
    return gemini_api_key


def create_chat_session(llm_model: str, temperature: float = 1.0):
    """Crea y retorna una sesión de chat utilizando el modelo especificado."""
    generation_config = {
        "temperature": temperature,
        "top_p": 0.95,
        "top_k": 40,
        "max_output_tokens": 8192,
        "response_mime_type": "text/plain",
    }
    model = genai.GenerativeModel(model_name=llm_model, generation_config=generation_config)
    return model.start_chat(history=[])

def read_problems_from_csv(input_filename: str):
    """Lee el archivo CSV y retorna una lista de problemas."""
    problems = []
    with open(input_filename, mode='r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)
        next(csv_reader, None)  # Salta la cabecera si existe
        for fields in csv_reader:
            if len(fields) < 2:
                continue
            problem = {"ID": fields[0], "Description": fields[1]}
            problems.append(problem)
    return problems



def extract_code(response_text: str) -> str:
    """
    Extrae el código Python del texto de respuesta, eliminando bloques markdown.
    Si se encuentra un bloque '```python' o '```', se extrae el contenido interno.
    """
    if "```python" in response_text:
        parts = response_text.split("```python")
        if len(parts) > 1:
            code_content = parts[1].split("```")[0].strip()
            return code_content
    elif "```" in response_text:
        parts = response_text.split("```")
        if len(parts) > 1:
            return parts[1].strip()
    return response_text.strip()


def process_problem(problem: dict, index: int, chat_session, llm_model: str, temperature: float, output_directory: str) -> dict:
    """
    Procesa un problema:
      - Construye el prompt y envía la solicitud al modelo.
      - Extrae y limpia el código Python de la respuesta.
      - Guarda el código en un archivo.
      - Retorna un diccionario con las métricas y el resultado.
    """
    prompt_text = (
        f"Write a Python solution for the following problem:\n\n"
        f"{problem['Description']}\n\n"
        "Provide only executable Python code, no explanations."
    )
    response = chat_session.send_message(prompt_text)
    python_code = extract_code(response.text)
    
    # Guarda el código generado en un archivo
    output_file = Path(output_directory) / f"output_{index + 1}.py"
    with open(output_file, mode='w', encoding='utf-8') as f:
        f.write(python_code)
    print(f"Python code saved successfully in {output_file}.")
    
    # Métricas del código
    code_metrics = {
        "problem_id": index + 1,
        "code_length": len(python_code),
        "model": llm_model,
        "temperature": temperature,
        "output_tokens": response.candidates[0].token_count if hasattr(response, 'candidates') else 0,
        "input_tokens": len(prompt_text.split())
    }
    
    return {
        "ID": index + 1,
        "code": python_code,
        "result": "SUCCESS",
        "true_count": code_metrics["code_length"],  # Se utiliza la longitud del código como métrica (ajustable)
        "false_count": 0
    }


def main():
    try:
        # Configuración inicial y creación de la sesión de chat
        setup_environment()
        llm_model = "gemini-2.0-pro-exp-02-05"
        temperature_value = 0  # Valor usado para organizar los directorios de salida
        chat_session = create_chat_session(llm_model, temperature=temperature_value)
        
        # Configuración de rutas y parámetros
        input_filename = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\data\processed\leetcode_problems_processed_data.csv"
        output_directory = rf"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\py_files_outputs_v2\temperature-{temperature_value}\{llm_model}"
        results_directory = rf"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results\temperature-{temperature_value}\{llm_model}"
        
        # Crear directorios si no existen
        Path(output_directory).mkdir(parents=True, exist_ok=True)
        Path(results_directory).mkdir(parents=True, exist_ok=True)
        
        # Lectura de los problemas
        problems = read_problems_from_csv(input_filename)
        if not problems:
            print("No se encontraron problemas en el archivo CSV.")
            return
        
        start_index = 0
        max_problems = 15
        results = []
        
        # Procesa cada problema
        for i in range(start_index, min(len(problems), start_index + max_problems)):
            print(f"Processing Problem {i + 1} of {max_problems}")
            start_time = time.time()
            try:
                result = process_problem(problems[i], i, chat_session, llm_model, temperature_value, output_directory)
                results.append(result)
            except Exception as e:
                print(f"Error processing problem {i + 1}: {e}")
                results.append({
                    "ID": i + 1,
                    "code": "",
                    "result": str(e),
                    "true_count": "ERROR",
                    "false_count": "ERROR"
                })
            elapsed_time = time.time() - start_time
            # Pausa para respetar el límite de tasa: 7 segundos por iteración
            if i < min(len(problems), start_index + max_problems) - 1:
                time.sleep(max(0, 60 - elapsed_time))
        
        # Guardar resultados en un archivo CSV
        results_csv = Path(results_directory) / f"results_{llm_model}.csv"
        with open(results_csv, mode='w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=["ID", "code", "result", "true_count", "false_count"])
            writer.writeheader()
            writer.writerows(results)
        print(f"CSV file '{results_csv}' created successfully.")
    
    except Exception as ex:
        print(f"An error occurred: {ex}")


if __name__ == "__main__":
    main()
